In [5]:
import torch

from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

The torchvision.datasets module contains Dataset objects for many real-world vision data 

In [6]:
# Download training data from open datasets; (imgs the model learns from)
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets. (imgs used later to check how well it learned)
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

___
We pass the Dataset as an argument to DataLoader. This wraps an iterable over our dataset, and supports automatic batching, sampling, shuffling and multiprocess data loading. 

Here we define a batch size of 64, i.e. each element in the dataloader iterable will return a batch of 64 features and labels.

In [7]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for x,y in test_dataloader:
    print(f"Shape of x [N, C, H, W]: {x.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break   # Shows only the first two lines.

Shape of x [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


___
### Creating Models

To define a neural network in PyTorch, we create a class that inherits from nn.Module. We define the layers of the network in the __init__ function and specify how data will pass through the network in the **forward** function. 

To accelerate operations in the neural network, we move it to the accelerator such as CUDA, MPS, MTIA, or XPU. If the current accelerator is available, we will use it. Otherwise, we use the CPU.

In [8]:
# Use an accelerator (e.g. GPU) if available; otherwise use the CPU.
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")


# Define the neural network architecture.
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        # Flatten each 28×28 image into 784 values.
        self.flatten = nn.Flatten()

        # Define the sequence of layers in the network.
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),  # 784 inputs → 512 neurons
            nn.ReLU(),              # Add nonlinearity
            nn.Linear(512, 512),    # 512 → 512 neurons
            nn.ReLU(),              # Add nonlinearity
            nn.Linear(512, 10)      # 512 → 10 class scores
        )

    # Define how an input passes through the network.
    def forward(self, x):
        x = self.flatten(x)                 # 28×28 → 784
        logits = self.linear_relu_stack(x)  # Pass through all layers
        return logits                       # Return the 10 output scores


# Create the model and move it to the selected device.
model = NeuralNetwork().to(device)

# Display the model architecture.
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


___
### Optimizing the Model Parameters

To train a model, we need a **loss function** and an **optimizer**.

In [9]:
# Loss: "How badly did I mess up?"
loss_fn = nn.CrossEntropyLoss()
# Optimizer: "Okay, how do I change my parameters to mess up less next time?"
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In a single training loop, the model makes predictions on the training dataset (fed to it in batches), and backpropagates the prediction error to adjust the model’s parameters.

In [10]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)  # Total number of training examples
    model.train()                   # Put the model in training mode

    # X = the input data, the images.
    # y = the correct answers (labels) for those images.
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)  # Move data to the selected device

        # Make predictions and calculate the prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagate the error and update the model's parameters
        loss.backward()          # Calculate gradients
        optimizer.step()         # Adjust weights using the gradients
        optimizer.zero_grad()    # Clear gradients before the next batch

        # Print the loss and training progress every 100 batches
        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

We also check the model’s performance against the test dataset to ensure it is learning.

In [11]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)       # Total number of test examples
    num_batches = len(dataloader)       # Number of test batches

    model.eval()                        # Put the model in evaluation mode
    test_loss, correct = 0, 0            # Track total loss and correct predictions

    with torch.no_grad():                # Don't calculate gradients while testing
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)  # Move data to the selected device

            pred = model(X)              # Make predictions for this batch

            # Add this batch's loss to the total test loss
            test_loss += loss_fn(pred, y).item()

            # Count how many predictions match the correct labels
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches             # Calculate average loss
    correct /= size                      # Calculate accuracy

    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

The training process is conducted over several iterations (epochs). During each epoch, the model learns parameters to make better predictions. 

We print the model’s accuracy and loss at each epoch; we’d like to see the accuracy increase and the loss decrease with every epoch.

In [12]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n--------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
--------------
loss: 2.301615  [   64/60000]
loss: 2.296146  [ 6464/60000]
loss: 2.275724  [12864/60000]
loss: 2.267960  [19264/60000]
loss: 2.256288  [25664/60000]
loss: 2.232913  [32064/60000]
loss: 2.232531  [38464/60000]
loss: 2.198757  [44864/60000]
loss: 2.196023  [51264/60000]
loss: 2.167220  [57664/60000]
Test Error: 
 Accuracy: 44.2%, Avg loss: 2.163013 

Epoch 2
--------------
loss: 2.173651  [   64/60000]
loss: 2.167063  [ 6464/60000]
loss: 2.110356  [12864/60000]
loss: 2.121314  [19264/60000]
loss: 2.075365  [25664/60000]
loss: 2.027378  [32064/60000]
loss: 2.037423  [38464/60000]
loss: 1.965116  [44864/60000]
loss: 1.972481  [51264/60000]
loss: 1.890901  [57664/60000]
Test Error: 
 Accuracy: 54.2%, Avg loss: 1.896398 

Epoch 3
--------------
loss: 1.931159  [   64/60000]
loss: 1.904893  [ 6464/60000]
loss: 1.788259  [12864/60000]
loss: 1.819200  [19264/60000]
loss: 1.713971  [25664/60000]
loss: 1.675451  [32064/60000]
loss: 1.673846  [38464/60000]
loss: 1.582884  [